# Montana Young Child Credit Reform Analysis (2027)

This notebook analyzes the impact of an expanded Montana child credit for children under 6.

## Reform Details
- **Credit Amount**: $1,000 per qualifying child under age 6
- **Eligibility**: Children under 6 years old who are dependents with SSN
- **Phase-out thresholds**:
  - Joint/Surviving Spouse: $80,000 AGI
  - Single/Head of Household/Separate: $40,000 AGI
- **Phase-out**: $50 reduction per $1,000 of income above threshold
- **Refundable**: Yes

## Analysis Goals
1. Count children under age 6 in Montana
2. Count children benefiting from the reform
3. Calculate budgetary impact
4. Measure poverty impacts

In [1]:
from policyengine_us import Microsimulation
from policyengine_core.reforms import Reform
import pandas as pd
import numpy as np

MT_DATASET = "hf://policyengine/policyengine-us-data/states/MT.h5"
PERIOD = 2027
AGE_LIMIT = 6

## Define Reform

The reform enables the Montana child credit with expanded age limit (under 6) and modified phase-out thresholds for 2027.

In [2]:
def create_young_child_credit_reform():
    """Enable Montana young child credit reform for 2027."""
    reform = Reform.from_dict(
        {
            "gov.contrib.states.mt.newborn_credit.age_limit": {
                "2027-01-01.2100-12-31": 6
            },
            "gov.contrib.states.mt.newborn_credit.in_effect": {
                "2027-01-01.2100-12-31": True
            },
            "gov.contrib.states.mt.newborn_credit.reduction.threshold.JOINT": {
                "2027-01-01.2100-12-31": 80000
            },
            "gov.contrib.states.mt.newborn_credit.reduction.threshold.SINGLE": {
                "2027-01-01.2100-12-31": 40000
            },
            "gov.contrib.states.mt.newborn_credit.reduction.threshold.SEPARATE": {
                "2027-01-01.2100-12-31": 40000
            },
            "gov.contrib.states.mt.newborn_credit.reduction.threshold.SURVIVING_SPOUSE": {
                "2027-01-01.2100-12-31": 80000
            },
            "gov.contrib.states.mt.newborn_credit.reduction.threshold.HEAD_OF_HOUSEHOLD": {
                "2027-01-01.2100-12-31": 40000
            }
        },
        country_id="us",
    )
    return reform

print("Reform function defined!")

Reform function defined!


## Load Simulations

In [3]:
print("Loading baseline (current law - no young child credit)...")
baseline = Microsimulation(dataset=MT_DATASET)
print("Baseline loaded")

print("\nLoading reform (young child credit enabled)...")
reform = create_young_child_credit_reform()
reformed = Microsimulation(dataset=MT_DATASET, reform=reform)
print("Reform loaded")

print("\n" + "="*60)
print("All simulations ready!")
print("="*60)

Loading baseline (current law - no young child credit)...
Baseline loaded

Loading reform (young child credit enabled)...
Reform loaded

All simulations ready!


## Count Children Under Age 6

In [4]:
# Get age and dependent status (MicroSeries with embedded weights)
age = baseline.calc("age", period=PERIOD)
is_dependent = baseline.calc("is_tax_unit_dependent", period=PERIOD)

# Children under 6 - MicroSeries handles weighting automatically
is_under_6 = age < AGE_LIMIT
children_under_6 = is_under_6.sum()

# Dependent children under 6
dependent_under_6 = is_under_6 & is_dependent
dependent_children_under_6 = dependent_under_6.sum()

# Total population for percentage calculations
total_population = baseline.calc("person_weight", period=PERIOD).sum()

print("="*60)
print(f"CHILDREN UNDER AGE {AGE_LIMIT} IN MONTANA ({PERIOD})")
print("="*60)
print(f"Total children under age {AGE_LIMIT}:      {children_under_6:,.0f}")
print(f"Dependent children under age {AGE_LIMIT}:  {dependent_children_under_6:,.0f}")
print("="*60)

CHILDREN UNDER AGE 6 IN MONTANA (2027)
Total children under age 6:      65,366
Dependent children under age 6:  65,366


## Children Benefiting from Reform

We measure how many children under 6 are in households that gain from the reform.

In [5]:
# Calculate household-level income change (MicroSeries handles weighting)
baseline_income = baseline.calc("household_net_income", period=PERIOD, map_to="person")
reformed_income = reformed.calc("household_net_income", period=PERIOD, map_to="person")
income_change = reformed_income - baseline_income

# Children under 6 who benefit (in households gaining more than $1)
children_benefiting_mask = is_under_6 & (income_change > 1)
children_benefiting = children_benefiting_mask.sum()

# Dependent children under 6 who benefit
dependent_children_benefiting_mask = dependent_under_6 & (income_change > 1)
dependent_children_benefiting = dependent_children_benefiting_mask.sum()

# Average benefit for children who benefit
if children_benefiting > 0:
    avg_benefit_children = (income_change * children_benefiting_mask).sum() / children_benefiting
else:
    avg_benefit_children = 0

print("="*60)
print(f"CHILDREN BENEFITING FROM YOUNG CHILD CREDIT ({PERIOD})")
print("="*60)
print(f"Children under {AGE_LIMIT} benefiting:           {children_benefiting:,.0f}")
print(f"Dependent children under {AGE_LIMIT} benefiting: {dependent_children_benefiting:,.0f}")
print(f"Percentage of children under {AGE_LIMIT}:        {children_benefiting / children_under_6 * 100:.1f}%")
print(f"Average benefit per benefiting child:  ${avg_benefit_children:,.0f}")
print("="*60)

CHILDREN BENEFITING FROM YOUNG CHILD CREDIT (2027)
Children under 6 benefiting:           31,606
Dependent children under 6 benefiting: 31,606
Percentage of children under 6:        48.4%
Average benefit per benefiting child:  $1,401


## Budgetary Impact

In [6]:
# Calculate total cost of the reform
baseline_total = baseline.calc("household_net_income", period=PERIOD, map_to="household").sum()
reformed_total = reformed.calc("household_net_income", period=PERIOD, map_to="household").sum()
total_cost = reformed_total - baseline_total

print("="*60)
print(f"BUDGETARY IMPACT ({PERIOD})")
print("="*60)
print(f"Total cost of young child credit:  ${total_cost/1e6:,.2f} million")
print(f"Cost per benefiting child:         ${total_cost / children_benefiting:,.0f}")
print("="*60)

BUDGETARY IMPACT (2027)
Total cost of young child credit:  $25.50 million
Cost per benefiting child:         $807


## Poverty Impact

In [7]:
def calculate_poverty_metrics(sim, period, child_only=False):
    """Calculate poverty metrics using MicroSeries."""
    age = sim.calc("age", period=period)
    in_poverty = sim.calc("person_in_poverty", period=period)
    
    if child_only:
        mask = age < 18
        poverty_rate = (in_poverty * mask).sum() / mask.sum()
        people_in_poverty = (in_poverty * mask).sum()
    else:
        poverty_rate = in_poverty.mean()
        people_in_poverty = in_poverty.sum()
    
    return poverty_rate, people_in_poverty

# Overall poverty
baseline_poverty_rate, baseline_poverty_count = calculate_poverty_metrics(baseline, PERIOD)
reformed_poverty_rate, reformed_poverty_count = calculate_poverty_metrics(reformed, PERIOD)

# Child poverty
baseline_child_poverty_rate, baseline_child_poverty_count = calculate_poverty_metrics(baseline, PERIOD, child_only=True)
reformed_child_poverty_rate, reformed_child_poverty_count = calculate_poverty_metrics(reformed, PERIOD, child_only=True)

print("="*60)
print(f"POVERTY IMPACT ({PERIOD})")
print("="*60)
print("\nOVERALL POVERTY:")
print(f"  Baseline poverty rate: {baseline_poverty_rate*100:.2f}%")
print(f"  Reformed poverty rate: {reformed_poverty_rate*100:.2f}%")
print(f"  Change:                {(reformed_poverty_rate - baseline_poverty_rate)*100:.2f} pp")
print(f"  People lifted:         {baseline_poverty_count - reformed_poverty_count:,.0f}")

print("\nCHILD POVERTY (under 18):")
print(f"  Baseline child poverty rate: {baseline_child_poverty_rate*100:.2f}%")
print(f"  Reformed child poverty rate: {reformed_child_poverty_rate*100:.2f}%")
print(f"  Change:                      {(reformed_child_poverty_rate - baseline_child_poverty_rate)*100:.2f} pp")
print(f"  Children lifted:             {baseline_child_poverty_count - reformed_child_poverty_count:,.0f}")
print("="*60)

POVERTY IMPACT (2027)

OVERALL POVERTY:
  Baseline poverty rate: 14.19%
  Reformed poverty rate: 13.98%
  Change:                -0.20 pp
  People lifted:         2,367

CHILD POVERTY (under 18):
  Baseline child poverty rate: 12.84%
  Reformed child poverty rate: 12.35%
  Change:                      -0.49 pp
  Children lifted:             1,185


## Winners and Losers

In [8]:
# Person-level analysis (MicroSeries handles weighting)
person_income_change = reformed.calc("household_net_income", period=PERIOD, map_to="person") - \
                       baseline.calc("household_net_income", period=PERIOD, map_to="person")

winners = (person_income_change > 1).sum()
losers = (person_income_change < -1).sum()
unchanged = ((person_income_change >= -1) & (person_income_change <= 1)).sum()

print("="*60)
print(f"WINNERS AND LOSERS ({PERIOD})")
print("="*60)
print(f"Winners (gain > $1):    {winners:,.0f} ({winners/total_population*100:.1f}%)")
print(f"Losers (lose > $1):     {losers:,.0f} ({losers/total_population*100:.1f}%)")
print(f"Unchanged:              {unchanged:,.0f} ({unchanged/total_population*100:.1f}%)")
print(f"\nAverage gain (winners): ${(person_income_change * (person_income_change > 1)).sum() / winners:,.0f}")
print("="*60)

WINNERS AND LOSERS (2027)
Winners (gain > $1):    100,388 (0.1%)
Losers (lose > $1):     0 (0.0%)
Unchanged:              1,056,049 (0.8%)

Average gain (winners): $1,240


## Summary

In [9]:
print("\n" + "="*70)
print("MONTANA YOUNG CHILD CREDIT REFORM SUMMARY")
print("="*70)
print(f"\nYear: {PERIOD}")
print(f"\nReform: $1,000 refundable credit per child under age {AGE_LIMIT}")
print(f"        Phase-out starts at $40k (single) / $80k (joint) AGI")
print("\n" + "-"*70)
print("KEY FINDINGS:")
print("-"*70)
print(f"  Children under age {AGE_LIMIT} in Montana:     {children_under_6:,.0f}")
print(f"  Children benefiting from reform:     {children_benefiting:,.0f}")
print(f"  Share of young children benefiting:  {children_benefiting / children_under_6 * 100:.1f}%")
print(f"\n  Total budgetary cost:                ${total_cost/1e6:,.2f} million")
print(f"  Average benefit per child:           ${total_cost / children_benefiting:,.0f}")
print(f"\n  People gaining income:               {winners:,.0f} ({winners/total_population*100:.1f}% of population)")
print(f"  Overall poverty reduction:           {(baseline_poverty_count - reformed_poverty_count):,.0f} people")
print(f"  Child poverty reduction:             {(baseline_child_poverty_count - reformed_child_poverty_count):,.0f} children")
print("="*70)


MONTANA YOUNG CHILD CREDIT REFORM SUMMARY

Year: 2027

Reform: $1,000 refundable credit per child under age 6
        Phase-out starts at $40k (single) / $80k (joint) AGI

----------------------------------------------------------------------
KEY FINDINGS:
----------------------------------------------------------------------
  Children under age 6 in Montana:     65,366
  Children benefiting from reform:     31,606
  Share of young children benefiting:  48.4%

  Total budgetary cost:                $25.50 million
  Average benefit per child:           $807

  People gaining income:               100,388 (0.1% of population)
  Overall poverty reduction:           2,367 people
  Child poverty reduction:             1,185 children


## Export Results

In [10]:
results = {
    "Metric": [
        f"Children under age {AGE_LIMIT}",
        "Children benefiting",
        "Share benefiting",
        "Total cost",
        "Average benefit per child",
        "People gaining income",
        "Share of population gaining",
        "Overall poverty reduction",
        "Child poverty reduction"
    ],
    "Value": [
        f"{children_under_6:,.0f}",
        f"{children_benefiting:,.0f}",
        f"{children_benefiting / children_under_6 * 100:.1f}%",
        f"${total_cost/1e6:,.2f}M",
        f"${total_cost / children_benefiting:,.0f}",
        f"{winners:,.0f}",
        f"{winners/total_population*100:.1f}%",
        f"{(baseline_poverty_count - reformed_poverty_count):,.0f} people",
        f"{(baseline_child_poverty_count - reformed_child_poverty_count):,.0f} children"
    ]
}

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

df_results.to_csv("mt_young_child_credit_results.csv", index=False)
print("\nExported to: mt_young_child_credit_results.csv")

                     Metric          Value
       Children under age 6         65,366
        Children benefiting         31,606
           Share benefiting          48.4%
                 Total cost        $25.50M
  Average benefit per child           $807
      People gaining income        100,388
Share of population gaining           0.1%
  Overall poverty reduction   2,367 people
    Child poverty reduction 1,185 children

Exported to: mt_young_child_credit_results.csv
